# CPA Screening — End-to-End Runner

This notebook runs the whole Phase 1 Day 1 pipeline on Colab:
1. Mount Drive (optional persistent cache)
2. Clone the repo and install base deps
3. Sanity-check imports BEFORE running the pipeline
4. Build the consolidated dataset + print audit
5. Train the Random Forest baseline
6. Display results and zip artifacts for download

Phase 1 Day 1 only includes the RF baseline against DOLMEN + hand-curated Higgins data. ChemBERTa+LoRA arrives in Day 2 after PAUSE POINT 1 — its deps live in `requirements-deep.txt` and aren't installed here.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d cpa-screening ] || git clone https://github.com/cmendoza1031/cpa-screening.git
%cd /content/cpa-screening
!git pull --ff-only

In [ ]:
# Install Phase 1 Day 1 base deps. NO --quiet so any failure is visible.
# We intentionally do NOT install requirements-deep.txt yet (torch / deepchem /
# transformers / peft) because deepchem in particular conflicts with Colab's
# pre-installed environment and can cause silent partial-install failures.
!pip install -r requirements.txt

In [ ]:
# Sanity check — fail LOUDLY here rather than silently inside the pipeline.
import importlib, sys
REQUIRED = ['rdkit', 'pubchempy', 'pandas', 'numpy', 'sklearn', 'scipy', 'matplotlib', 'requests', 'pyarrow']
missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f'  ok  {mod}')
    except ImportError as e:
        print(f'  MISSING  {mod}: {e}')
        missing.append(mod)
if missing:
    raise RuntimeError(f'Missing required modules: {missing}. Re-run the pip install cell, then retry.')
print('\nAll Phase 1 Day 1 deps available.')

In [ ]:
# Optional: persist data/results across Colab sessions via Drive.
import os, pathlib
DRIVE_CACHE = pathlib.Path('/content/drive/MyDrive/cpa-screening')
if DRIVE_CACHE.parent.exists():
    DRIVE_CACHE.mkdir(exist_ok=True)
    for sub in ('data', 'results'):
        src = pathlib.Path(sub)
        dst = DRIVE_CACHE / sub
        dst.mkdir(parents=True, exist_ok=True)
        if src.is_symlink():
            src.unlink()
    print('Drive cache available at', DRIVE_CACHE)
else:
    print('Drive not mounted; using ephemeral storage')

## 2. Build the dataset

First run will download DOLMEN raw CSVs (instant) and map FDA IID CAS numbers to SMILES via PubChem (slow first run, ~1-3 min for ~1.5k entries; cached after).

Higgins CSVs are already in the repo (`data/raw/higgins_*.csv`); no manual transcription needed.

In [ ]:
# --skip-tox21 because DeepChem isn't in the Day 1 deps; uncomment --skip-fda
# if you want a fast smoke run without FDA IID candidate scoring.
!python -m src.data --skip-tox21
# !python -m src.data --skip-tox21 --skip-fda

In [ ]:
import json, pathlib
audit = json.loads(pathlib.Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2)[:2000])

## 3. Train Random Forest baseline

In [ ]:
!python -m src.train --model rf --seed 0

In [ ]:
import pandas as pd
df = pd.read_csv('results/results_table.csv')
df

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('results/figures').glob('*.png')):
    print(p)
    display(Image(str(p)))

## 4. Bundle results for download

In [ ]:
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip

In [ ]:
from google.colab import files
files.download('results.zip')

---
## PAUSE POINT 1

Report back to the developer with:
- Compound counts per task from the audit cell above (look at `unique_compounds` and `per_task`)
- RF metrics from `results_table.csv` (focus on val rows; test rows are noisy on Higgins tasks because n is small)
- Any errors or PubChem failure counts (`audit.pubchem_cache.n_misses`, plus `data/.cache/pubchem_failures.txt` contents)

Phase 1 Day 2 (ChemBERTa+LoRA) starts after confirmation; that day's setup cell will install `requirements-deep.txt`.